In [1]:
import os
import pandas as pd

# folder = "loocv_2_stg_model"
# out = "model_out_loocv_2_stage"

folder = "loocv_2_stg_model_geo"
out = "model_out_loocv_2_stage_geo"

path = f"/data/big/fmoss/data/model_output/{folder}"
dfs = []
for file_name in os.listdir(path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(path, file_name)
        df = pd.read_csv(file_path)
        dfs.append(df)
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

In [2]:
os.makedirs("/data/big/fmoss/data/model_output/merged/", exist_ok=True)
combined_df.to_parquet(f"/data/big/fmoss/data/model_output/merged/{out}.parquet")

To adm1

In [ ]:
import os
import pandas as pd

out = "model_out_loocv_2_stage_geo"
combined_df = pd.read_parquet(f"/data/big/fmoss/data/model_output/merged/{out}.parquet")

In [4]:
def adm1_predictions(df):
    """
    Aggregate grid-level ensemble predictions to ADM1 level.
    Handles mean, ±std, max, and wind-model predictions using population weighting.
    """
    df = df.copy()
    group_keys = ["DisNo.", "GID_1"]

    df["reported"] = df["perc_affected_pop_grid_region"].copy()
    # df["reported"] = df["actual_perc"].copy()
    df["prediction_perc"] = df["prediction_perc"].clip(0, 100)
    df["prediction_ppl"] = df["prediction_perc"] * df["population"] / 100


    # Aggregate by ADM1
    agg_dict = {
    "prediction_ppl": ("prediction_ppl", "sum"),
    "reported": ("reported", "max"),
    "population": ("population", "sum"),
    "wind_speed": ("wind_speed", "max"),
    "year": ("year", "max"),
    }

    # Add date only if present
    if "date" in df.columns:
        agg_dict["date"] = ("date", "max")

    grouped = (
        df.groupby(group_keys)
        .agg(**agg_dict)
        .reset_index()
    )

    # Convert back to percentages at ADM1 level
    grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

    # Clip to valid percentages
    for col in ["predicted", "reported"]:
        grouped[col] = grouped[col].clip(0, 100)

    # Compute errors
    grouped["error"] = grouped["predicted"] - grouped["reported"]
    grouped["abs_error"] = grouped["error"].abs()

    return grouped

df_adm1 = adm1_predictions(combined_df)

In [5]:
df_adm1[(df_adm1["DisNo."] == "2018-0341-PHL") & (df_adm1.GID_1 == "PHL.35_1")].sort_values("reported")

,DisNo.,GID_1,prediction_ppl,reported,population,wind_speed,year,date,predicted,error,abs_error
24419,2018-0341-PHL,PHL.35_1,106220.716219,100.0,769234.129997,48.830179,2018,2018-09-16,13.808633,-86.191367,86.191367


In [7]:
# combined_df[(combined_df["DisNo."] == "2018-0341-PHL") & (combined_df.GID_1 == "PHL.35_1")].sort_values("perc_affected_pop_grid_region")

In [8]:
# out_adm1 = "2_stage_model_loocv"
# out_adm1 = "model_out_loocv_2_stg_model_adm1"
out_adm1 = "model_out_loocv_2_stg_model_geo_adm1"
df_adm1.dropna(ignore_index=True).to_parquet(f"/data/big/fmoss/data/model_output/merged/adm1_grouped/{out_adm1}.parquet")